# 02 - Segmentation, Churn & Revenue Analysis
Banking Customer Analytics Project

This notebook builds customer segments, analyzes churn drivers, estimates revenue, and surfaces the business insights used in the final report and dashboard.

In [1]:
import sys
sys.path.append('..')

import pandas as pd

from src.segmentation import build_segments, segment_summary_table
from src.churn_analysis import full_churn_report, RETENTION_RECOMMENDATIONS
from src.revenue_analysis import full_revenue_report

pd.set_option('display.max_columns', None)

In [2]:
clean_df = pd.read_csv('../data/processed/bank_customers_clean.csv')
clean_df.shape

(10000, 13)

## Customer Segmentation
See `src/segmentation.py` docstrings for the full business logic behind each segment.

In [3]:
segmented_df = build_segments(clean_df)
segmented_df.to_csv('../data/processed/bank_customers_segmented.csv', index=False)
segment_summary_table(segmented_df)

2026-07-04 11:23:15 | INFO     | src.segmentation | Building customer segments


2026-07-04 11:23:15 | INFO     | src.segmentation | Value segment distribution:
ValueSegment
High-Value      6180
Medium-Value    3820
Name: count, dtype: int64


2026-07-04 11:23:15 | INFO     | src.segmentation | Life-stage distribution:
LifeStageSegment
Established           6305
Senior                3493
Young Professional     202
Name: count, dtype: int64


2026-07-04 11:23:15 | INFO     | src.segmentation | High-risk customers: 2458 (24.6%)


2026-07-04 11:23:15 | INFO     | src.segmentation | Premium customers: 394 (3.9%)


,ValueSegment,customers,avg_balance,avg_credit_score,churn_rate,avg_products
0,High-Value,6180,78634.01,649.05,0.27,1.98
1,Medium-Value,3820,36231.76,647.95,0.41,1.00


In [4]:
segmented_df['LifeStageSegment'].value_counts()

LifeStageSegment
Established           6305
Senior                3493
Young Professional     202
Name: count, dtype: int64

In [5]:
segmented_df[['HighRiskChurn', 'PremiumCustomer']].mean().round(3) * 100

HighRiskChurn      24.6
PremiumCustomer     3.9
dtype: float64

## Churn Analysis

In [6]:
churn_report = full_churn_report(segmented_df)
print('Overall churn rate:', round(churn_report['overall_rate']*100, 2), '%')

2026-07-04 11:23:16 | INFO     | src.churn_analysis | Building full churn analysis report


2026-07-04 11:23:16 | INFO     | src.churn_analysis | Overall churn rate: 32.55%


Overall churn rate: 32.55 %


In [7]:
churn_report['by_geography']

,Geography,customers,churned,churn_rate
1,Germany,2458,987,0.4015
0,France,4987,1500,0.3008
2,Spain,2555,768,0.3006


In [8]:
churn_report['by_age_band']

,AgeBand,customers,churned,churn_rate
4,60+,3493,1580,0.4523
3,50-59,3049,925,0.3034
2,40-49,2689,617,0.2295
1,30-39,745,129,0.1732
0,18-29,24,4,0.1667


In [9]:
churn_report['by_products']

,NumOfProducts,customers,churned,churn_rate
0,1,5093,2042,0.4009
1,2,3986,1082,0.2715
3,4,201,40,0.1990
2,3,720,91,0.1264


In [10]:
churn_report['by_tenure']

,TenureBand,customers,churned,churn_rate
0,0-2 yrs,2756,937,0.3400
1,3-5 yrs,2672,869,0.3252
2,6-8 yrs,2759,877,0.3179
3,9+ yrs,1813,572,0.3155


In [11]:
churn_report['driver_ranking'].head(10)

,dimension,segment,customers,churn_rate,lift_vs_overall
0,LifeStageSegment,Senior,3493,0.4523,1.39
1,IsActiveMember,0,4737,0.4095,1.26
2,NumOfProducts,1,5093,0.4009,1.23
3,Geography,Germany,2458,0.4015,1.23
4,Gender,Female,4982,0.3374,1.04
5,HasCrCard,0,2944,0.3285,1.01
6,HasCrCard,1,7056,0.3243,1.00
7,Gender,Male,5018,0.3137,0.96
8,Geography,France,4987,0.3008,0.92
9,Geography,Spain,2555,0.3006,0.92


### Retention Recommendations

In [12]:
for rec in RETENTION_RECOMMENDATIONS:
    print('Driver:', rec['driver'])
    print(' Insight:', rec['insight'])
    print(' Recommendation:', rec['recommendation'])
    print()

Driver: Inactive membership
 Insight: Inactive members churn at roughly double the rate of active members.
 Recommendation: Launch a re-engagement campaign (app nudges, fee waivers, personal banker outreach) for members inactive 60+ days.

Driver: Single product ownership
 Insight: Customers holding only one product churn far more than those with 2 products.
 Recommendation: Prioritize cross-sell offers (savings account, credit card, investment product) to single-product customers in their first 12 months.

Driver: Germany geography
 Insight: German customers show a meaningfully higher churn rate than France or Spain.
 Recommendation: Commission a local NPS/satisfaction study in Germany; benchmark local competitor rates and fees.

Driver: Age 50+
 Insight: Churn rises steadily with age, peaking in the 50-59 band.
 Recommendation: Build a dedicated retirement/wealth-transition advisory track for customers approaching retirement age.

Driver: Zero-balance accounts
 Insight: Customers hol

## Revenue Analysis
See `src/revenue_analysis.py` for the documented revenue-proxy assumptions.

In [13]:
revenue_report, revenue_df = full_revenue_report(segmented_df)
revenue_df.to_csv('../data/processed/bank_customers_revenue.csv', index=False)
print('Total estimated annual revenue: $%.2f' % revenue_report['total_estimated_revenue'])

2026-07-04 11:23:16 | INFO     | src.revenue_analysis | Building full revenue analysis report


2026-07-04 11:23:16 | INFO     | src.revenue_analysis | Total estimated annual revenue: $17853014.53


Total estimated annual revenue: $17853014.53


In [14]:
revenue_report['by_country']

,Geography,customers,total_revenue,avg_revenue_per_customer
0,France,4987,8967120.68,1798.10
1,Spain,2555,4524420.36,1770.81
2,Germany,2458,4361473.49,1774.40


In [15]:
revenue_report['by_value_segment']

,ValueSegment,customers,total_revenue,avg_revenue_per_customer
0,High-Value,6180,13705755.88,2217.76
1,Medium-Value,3820,4147258.65,1085.67


In [16]:
revenue_report['by_age_group']

,AgeGroup,customers,total_revenue,avg_revenue_per_customer
0,60+,3493,6249413.93,1789.13
1,50-59,3049,5416621.47,1776.52
2,40-49,2689,4799035.60,1784.69
3,30-39,745,1352227.81,1815.07
4,18-29,24,35715.72,1488.16


In [17]:
revenue_report['top_segments']

,ValueSegment,LifeStageSegment,customers,total_revenue
0,High-Value,Established,3925,8691775.90
1,High-Value,Senior,2135,4747912.75
2,Medium-Value,Established,2380,2564155.45
3,Medium-Value,Senior,1358,1501501.18
4,High-Value,Young Professional,120,266067.23


## Summary
This notebook produced the analytics-ready dataset (`data/processed/bank_customers_revenue.csv`) that powers both the SQL database (`src/database.py`) and the Streamlit dashboard (`dashboard/app.py`). See `reports/business_insights.md` for the full list of 20+ data-backed business insights.